In [1]:
import pandas as pd
import numpy as np
import requests
import time

In [5]:
train_df = pd.read_excel("Gen_AI Dataset.xlsx", sheet_name=0)

print(f"Total rows: {len(train_df)}")
print(f"Columns: {list(train_df.columns)}")
train_df.head(3)

Total rows: 65
Columns: ['Query', 'Assessment_url']


,Query,Assessment_url
0,I am hiring for Java developers who can also c...,https://www.shl.com/solutions/products/product...
1,I am hiring for Java developers who can also c...,https://www.shl.com/solutions/products/product...
2,I am hiring for Java developers who can also c...,https://www.shl.com/solutions/products/product...


In [13]:
train_grouped = train_df.groupby("Query")["Assessment_url"].apply(list).reset_index()
train_grouped.columns = ["query", "relevant_urls"]

print(f"Total unique queries: {len(train_grouped)}")
print(f"\nQueries:")
for i, row in train_grouped.iterrows():
    print(f"\n{i+1}. {row['query'][:80]}...")
    print(f"   Relevant assessments: {len(row['relevant_urls'])}")

Total unique queries: 10

Queries:

1. Based on the JD below recommend me assessment for the Consultant position in my ...
   Relevant assessments: 5

2. Content Writer required, expert in English and SEO....
   Relevant assessments: 5

3. Find me 1 hour long assesment for the below job at SHL
Job Description

 Join a ...
   Relevant assessments: 9

4. I am hiring for Java developers who can also collaborate effectively with my bus...
   Relevant assessments: 5

5. I am looking for a COO for my company in China and I want to see if they are cul...
   Relevant assessments: 6

6. I want to hire a Senior Data Analyst with 5 years of experience and expertise in...
   Relevant assessments: 10

7. I want to hire new graduates for a sales role in my company, the budget is for a...
   Relevant assessments: 9

8. ICICI Bank Assistant Admin, Experience required 0-2 years, test should be 30-40 ...
   Relevant assessments: 6

9. KEY RESPONSIBITILES:

Manage the sound-scape of the station through a

In [17]:
def recall_at_k(recommended_urls, relevant_urls, k=10):
    recommended_urls = recommended_urls[:k]
    
    recommended_clean = [url.strip().rstrip("/") for url in recommended_urls]
    relevant_clean = [url.strip().rstrip("/") for url in relevant_urls]
    
    if len(relevant_clean) == 0:
        return 0.0
    
    found = len(set(recommended_clean) & set(relevant_clean))
    return found / len(relevant_clean)


def mean_recall_at_k(all_recommended, all_relevant, k=10):
    recalls = []
    for recommended, relevant in zip(all_recommended, all_relevant):
        r = recall_at_k(recommended, relevant, k)
        recalls.append(r)
    
    return np.mean(recalls), recalls

In [21]:
API_URL = "http://localhost:8000"

def get_predictions(query, top_k=10):
    try:
        response = requests.post(
            f"{API_URL}/recommend",
            json={"query": query, "top_k": top_k},
            timeout=60
        )
        if response.status_code == 200:
            data = response.json()
            urls = [a["url"] for a in data["recommended_assessments"]]
            return urls
        else:
            print(f" API Error: {response.status_code}")
            return []
    except Exception as e:
        print(f" Error: {e}")
        return []

In [37]:
all_recommended = []
all_relevant = []
query_results = []

for i, row in train_grouped.iterrows():
    query = row["query"]
    relevant_urls = row["relevant_urls"]
    
    print(f"Query {i+1}/{len(train_grouped)}: {query[:60]}...")
    
    predicted_urls = get_predictions(query, top_k=10)
    recall = recall_at_k(predicted_urls, relevant_urls, k=10)
    
    print(f"  Relevant: {len(relevant_urls)} | Predicted: {len(predicted_urls)} | Recall@10: {recall:.3f}")
    
    all_recommended.append(predicted_urls)
    all_relevant.append(relevant_urls)
    
    query_results.append({
        "query": query,
        "relevant_count": len(relevant_urls),
        "predicted_count": len(predicted_urls),
        "recall_at_10": recall
    })
    
    time.sleep(2)

mean_recall, individual_recalls = mean_recall_at_k(all_recommended, all_relevant, k=10)

print(f"\n{'='*50}")
print(f"Mean Recall@10: {mean_recall:.4f} ({mean_recall*100:.2f}%)")

Query 1/10: Based on the JD below recommend me assessment for the Consul...
  Relevant: 5 | Predicted: 6 | Recall@10: 0.000
Query 2/10: Content Writer required, expert in English and SEO....
  Relevant: 5 | Predicted: 7 | Recall@10: 0.200
Query 3/10: Find me 1 hour long assesment for the below job at SHL
Job D...
❌ API Error: 500
  Relevant: 9 | Predicted: 0 | Recall@10: 0.000
Query 4/10: I am hiring for Java developers who can also collaborate eff...
❌ API Error: 500
  Relevant: 5 | Predicted: 0 | Recall@10: 0.000
Query 5/10: I am looking for a COO for my company in China and I want to...
❌ API Error: 500
  Relevant: 6 | Predicted: 0 | Recall@10: 0.000
Query 6/10: I want to hire a Senior Data Analyst with 5 years of experie...
❌ API Error: 500
  Relevant: 10 | Predicted: 0 | Recall@10: 0.000
Query 7/10: I want to hire new graduates for a sales role in my company,...
❌ API Error: 500
  Relevant: 9 | Predicted: 0 | Recall@10: 0.000
Query 8/10: ICICI Bank Assistant Admin, Experience requ

In [39]:
results_df = pd.DataFrame(query_results)
results_df["recall_at_10"] = results_df["recall_at_10"].round(3)

#print(results_df.to_string(index=False))

print(f"\n--- SUMMARY ---")
print(f"Best Recall:    {results_df['recall_at_10'].max():.3f}")
print(f"Worst Recall:   {results_df['recall_at_10'].min():.3f}")
print(f"Mean Recall@10: {results_df['recall_at_10'].mean():.3f}")

results_df.to_csv("data/evaluation_results.csv", index=False)


--- SUMMARY ---
Best Recall:    0.200
Worst Recall:   0.000
Mean Recall@10: 0.020
